
## Message Trimming

Message trimming is the process of **removing older or less important messages** from the conversation before sending the context to the LLM.

### Purpose
- Prevent **context window overflow**
- Reduce token usage and cost
- Reduce latency
- Keep the most relevant context

### Example

```text
Before:
[Msg1, Msg2, Msg3, Msg4, Msg5, Msg6]

After trimming:
[        Msg3, Msg4, Msg5, Msg6]
````

> **Message trimming = Keep important/recent messages and remove unnecessary history.**

```

```

## Issue with Message Trimming

Message trimming removes older messages to stay within the LLM's context limit.

### Main Issue
- **Important information may be removed** along with old messages.
- The LLM may lose context needed to answer future questions.
- Long conversations can lose important decisions, instructions, or user details.

> **Problem:** Trimming reduces context size, but may also remove useful information.

> use **Summarization** to resolve this problem


In [8]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_core.prompts import PromptTemplate
from langgraph.checkpoint.memory import InMemorySaver
from langchain_google_genai import ChatGoogleGenerativeAI
from operator import add

# use for trimming messages
from langchain_core.messages.utils import trim_messages, count_tokens_approximately

In [2]:
llm = ChatGoogleGenerativeAI(
    model = "gemini-3.1-flash-lite",
    temperature=0.6
)


In [3]:
class State(TypedDict):
    messages: Annotated[list[BaseMessage], add]
    

In [13]:
MAX_TOKENS = 100

In [18]:

def chat_node(state:State):

    print("\n-----------\nTotal messages before triming: ", len(state["messages"]))

    # Get the complete conversation history from the current thread
    # and keep only the most recent messages within the token limit.
    # Note: All the messages will keep remains in the state, 
    # we are only fetching the latest MAX_TOKENS for llm

    messages = trim_messages(
        messages=state["messages"],
        strategy="last",
        token_counter=count_tokens_approximately, 
        max_tokens=MAX_TOKENS
    )

    print("after triming: message we are sharing with llm: ", len(messages))

    # Send the trimmed conversation to the LLM
    resp = llm.invoke(messages)

    # Add the LLM response back to the graph state.
    # The checkpointer can persist this updated state.
    return {
        "messages": [resp]
    }

In [15]:
checkpointer = InMemorySaver()

graph = StateGraph(State)\
    .add_node("chat_node", chat_node)\
    .add_edge(START, "chat_node")\
    .add_edge("chat_node", END)\
    .compile(checkpointer=checkpointer)

### To test trimming

Send enough messages to exceed your MAX_TOKENS:

Message 1
Message 2
Message 3
...
Message N

Then ask something that depends on an old message.

You should observe:
```markdown
PostgreSQL STM → contains conversation history
                    ↓
              trim_messages()
                    ↓
              Recent context
                    ↓
                   LLM
```

So you are testing two separate things:

Same thread → persistence + memory

Large history → trimming + limited LLM context

And importantly, trimming the messages sent to the LLM does not necessarily delete the older messages from PostgreSQL STM.

In [1]:
config = {
    "configurable": {
        "thread_id": "id_trim-1"
    }
}

while True:
    msg = input()

    if msg in ["done", "bye", "exit"]:
        break;
        
    resp = graph.invoke({"messages": [HumanMessage(content=msg)]}, config)

    print("\nmsg: ", msg)
    print(resp["messages"][-1])

In [19]:
# all the messages will keep in the checkpointer
checkpointer.get(config)

{'v': 4,
 'ts': '2026-08-21T15:06:41.809603+00:00',
 'id': '1f19d71e-a9b2-61c0-800d-687b9494ef43',
 'channel_versions': {'__start__': '00000000000000000000000000000014.0.28459015022938794',
  'messages': '00000000000000000000000000000015.0.842866062070132',
  'branch:to:chat_node': '00000000000000000000000000000015.0.842866062070132'},
 'versions_seen': {'__input__': {},
  '__start__': {'__start__': '00000000000000000000000000000013.0.41180381402553445'},
  'chat_node': {'branch:to:chat_node': '00000000000000000000000000000014.0.28459015022938794'}},
 'updated_channels': ['messages'],
 'channel_values': {'messages': [HumanMessage(content='my name is bhavin', additional_kwargs={}, response_metadata={}),
   AIMessage(content=[{'type': 'text', 'text': "Hello, Bhavin! It's nice to meet you. How are you doing today? Is there anything I can help you with?", 'extras': {'signature': 'EnEKbwERTTIPUuTgIFKz9pYFjwwETDkI38jJeiP64mIaqS141ce2++N/8l26BlILa/T4DQbmb/xUxMmBXkPhbZMgqqcAmDLSnKV83E/RJeGgL2c